In [ ]:
import pandas as pd
import json

from aloud_database.aloud_database import Database

In [ ]:
db = Database()
responses = pd.read_csv('data/lpm_responses.csv')

In [ ]:
query = """
    SELECT 
        c.id,
        c.conversion_raw_info->>'token' as token,
        c.conversion_raw_info
    FROM
        lead_tracking_prod.conversions c
    WHERE
        c.conversion_type_id = '2'
        AND c.campaign_id = 'ppt-lpm-cdf'
        AND c.conversion_raw_info->>'gender_score' is null
"""

responses_registered = db.execute_query(query=query)

In [ ]:
import logging

# Configuração básica do logger
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')

def get_response_by_token(token: str, registered_responses: pd.DataFrame):
    response_found = registered_responses[registered_responses['token'] == token]
    return response_found
    

def update_conversion_raw_info_by_id(conversion_id: str, conversion_raw_info: dict):
    query = f"""
        UPDATE
            lead_tracking_prod.conversions
        SET
            conversion_raw_info = '{json.dumps(conversion_raw_info)}'::jsonb
        where   
            id = '{conversion_id}'::uuid 
    """
    result = db.execute_update(query=query)

# Adiciona coluna de progresso se não existir
if 'processado' not in responses.columns:
    responses['processado'] = False

total = len(responses)
for _idx, _row in responses.iterrows():
    if _row.get('processado', False):
        continue  # Pula se já processado

    print(f"Processando {_idx+1}/{total}", end='\r')
    try:
        response_df = get_response_by_token(token=_row.get('token'), registered_responses=responses_registered)

        if not response_df.empty:
            campos_score = [
                ('score', 'new_score'),
                ('gender_score', 'gender_score'),
                ('age_score', 'age_score'),
                ('occupation_score', 'occupation_score'),
                ('income_score', 'income_score'),
                ('teb_score', 'teb_score')
            ]

            for idx, row in response_df.iterrows():
                try:
                    new_conversion_raw_info = row.get('conversion_raw_info').copy()
                    new_conversion_raw_info['reprocessed'] = True
                    for campo_destino, campo_origem in campos_score:
                        valor = _row.get(campo_origem)
                        if pd.notnull(valor):
                            new_conversion_raw_info[campo_destino] = int(valor)
                    update_conversion_raw_info_by_id(conversion_id=row['id'], conversion_raw_info=new_conversion_raw_info)
                except Exception as e:
                    logging.error(f"Erro ao atualizar conversão ID: {row.get('id', 'N/A')}. Detalhes: {e}")
        # Marca como processado independentemente do resultado
        responses.at[_idx, 'processado'] = True
    except Exception as e:
        logging.error(f"Erro ao processar linha {_idx+1} (token: {_row.get('token')}). Detalhes: {e}")
